In [1]:
# =========================================
# Classic ML pipeline on new paired dataset
# TF-IDF instead of spaCy embeddings in the paper 
# =========================================

import os
import re
import warnings
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore")

# =========================================
# 1) Path
# =========================================

NEW_DATA_PATH = "/kaggle/input/datasets/aabdollahii/humanvsai/dataset (1).xlsx"
print("New data path exists:", os.path.exists(NEW_DATA_PATH), NEW_DATA_PATH)

# =========================================
# 2) Load new paired dataset
# =========================================

df = pd.read_excel(NEW_DATA_PATH)
print("\nNew dataset shape:", df.shape)
print(df.head())
print(df.columns.tolist())

# =========================================
# 3) Utility functions
# =========================================

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"[\u200b-\u200f\uFEFF]", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def prepare_labels(y):
    y = np.array(y)
    if y.dtype == object:
        le = LabelEncoder()
        y_enc = le.fit_transform(y)
        return y_enc, le
    return y, None

def print_basic_info(df_long, text_col, label_col, name):
    print(f"\n===== {name} dataset info =====")
    print("Text column :", text_col)
    print("Label column:", label_col)
    print("Shape       :", df_long.shape)
    print("Null texts  :", df_long[text_col].isna().sum())
    print("Label dist  :")
    print(df_long[label_col].value_counts(dropna=False))

New data path exists: True /kaggle/input/datasets/aabdollahii/humanvsai/dataset (1).xlsx

New dataset shape: (1519, 10)
   year             filename  \
0  2003  HAM2-811011-027.ham   
1  2003  HAM2-811014-088.ham   
2  2003  HAM2-811015-093.ham   
3  2003  HAM2-811015-094.ham   
4  2003  HAM2-811016-002.ham   

                                                text  word_count  label  \
0  آغاز ساخت بدنه و سرريز بلندترين سد\nكشور\nعملي...         136  human   
1  گزارش حجم آب مفيد سدهاي تامين كننده آب\nشرب ته...          53  human   
2  كفاشيان: هر نامزدي به روند انتخابات \nاعتراض د...         138  human   
3  دعوت شدگان جديد در اردوي تيم واليبال \nجوانان ...         122  human   
4  نمايش آثار هنري دهه 40و50 جمهوري چك\nآثار هنري...         181  human   

                                        machine_text machine_label  \
0  آغاز فازهای کلیدی ساخت بزرگ‌ترین سازه آبی کشور...       machine   
1  وضعیت ذخایر آبی سدهای استراتژیک استان تهران که...       machine   
2  در فضای پرچالش ورزش کشو

In [2]:
# =========================================
# 4) Build long-format dataset (two samples per row)
# =========================================

df = df.reset_index().rename(columns={"index": "group_id"})

df["text_clean"] = df["text"].apply(clean_text)
df["machine_text_clean"] = df["machine_text"].apply(clean_text)

df = df[(df["text_clean"].str.len() > 0) & (df["machine_text_clean"].str.len() > 0)].reset_index(drop=True)
print("\nAfter cleaning, base shape:", df.shape)

human_part = df[["group_id", "year", "filename", "text_clean", "label"]].copy()
human_part = human_part.rename(columns={"text_clean": "text", "label": "target"})
human_part["source_type"] = "human"

machine_part = df[["group_id", "year", "filename", "machine_text_clean", "machine_label"]].copy()
machine_part = machine_part.rename(columns={"machine_text_clean": "text", "machine_label": "target"})
machine_part["source_type"] = "machine"

df_long = pd.concat([human_part, machine_part], ignore_index=True)

TEXT_COL = "text"
LABEL_COL = "target"
GROUP_COL = "group_id"

print("\nLong-format dataset shape:", df_long.shape)
print_basic_info(df_long, TEXT_COL, LABEL_COL, "Paired long-format")


After cleaning, base shape: (1519, 13)

Long-format dataset shape: (3038, 6)

===== Paired long-format dataset info =====
Text column : text
Label column: target
Shape       : (3038, 6)
Null texts  : 0
Label dist  :
target
human      1519
machine    1519
Name: count, dtype: int64


In [3]:
display(df_long)

,group_id,year,filename,text,target,source_type
0,0,2003,HAM2-811011-027.ham,آغاز ساخت بدنه و سرريز بلندترين سد كشور عمليات...,human,human
1,1,2003,HAM2-811014-088.ham,گزارش حجم آب مفيد سدهاي تامين كننده آب شرب تهر...,human,human
2,2,2003,HAM2-811015-093.ham,كفاشيان: هر نامزدي به روند انتخابات اعتراض دار...,human,human
3,3,2003,HAM2-811015-094.ham,دعوت شدگان جديد در اردوي تيم واليبال جوانان اي...,human,human
4,4,2003,HAM2-811016-002.ham,نمايش آثار هنري دهه 40و50 جمهوري چك آثار هنري ...,human,human
...,...,...,...,...,...,...
3033,1514,2007,HAM2-860216-029.ham,بیماری های مقاربتی در زندان ها صحت ندارد گروه ...,machine,machine
3034,1515,2007,HAM2-860217-018.ham,مسابقه جذاب با جوایز نقدی ویژه کدام مجموعه تلو...,machine,machine
3035,1516,2007,HAM2-860219-035.ham,دومین جشنواره بین المللی رسانه ای میراث فرهنگی...,machine,machine
3036,1517,2007,HAM2-860220-076.ham,ایجاد مراکز تفکر و اندیشه ورزی در سطح محلات، ی...,machine,machine


In [4]:
# Encode labels to 0/1
y, le = prepare_labels(df_long[LABEL_COL].values)
groups = df_long[GROUP_COL].values
texts = df_long[TEXT_COL].values

print("\nLabel mapping (class -> id):")
for cls, idx in zip(le.classes_, range(len(le.classes_))):
    print(f"  {cls} -> {idx}")

# =========================================
# 5) Group-aware train/test split (avoid leakage)
# =========================================

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(texts, y, groups=groups))

X_train_text = texts[train_idx]
X_test_text = texts[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

print("\nTrain/test sizes (group-aware):")
print("Train:", len(X_train_text), "Test:", len(X_test_text))


Label mapping (class -> id):
  human -> 0
  machine -> 1

Train/test sizes (group-aware):
Train: 2430 Test: 608


In [5]:
# =========================================
# 6) Models (TF-IDF + classifier)
# =========================================

def build_models():
    tfidf = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        lowercase=False  # Persian: keep as-is
    )

    models = {
        "TFIDF+MultinomialNB": Pipeline([
            ("tfidf", tfidf),
            ("clf", MultinomialNB(alpha=1.0))
        ]),

        "TFIDF+LinearSVC": Pipeline([
            ("tfidf", clone(tfidf)),
            ("clf", LinearSVC(C=1.0))
        ]),

        "TFIDF+RandomForest": Pipeline([
            ("tfidf", clone(tfidf)),
            ("clf", RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                random_state=42,
                n_jobs=-1
            ))
        ]),

        "TFIDF+KNN": Pipeline([
            ("tfidf", clone(tfidf)),
            ("clf", KNeighborsClassifier(
                n_neighbors=15,
                metric="cosine"
            ))
        ]),
    }
    return models

# =========================================
# 7) Evaluation
# =========================================

def evaluate_model(model, X_train_text, y_train, X_test_text, y_test, name="model"):
    model.fit(X_train_text, y_train)

    y_pred = model.predict(X_test_text)

    # Some models provide decision_function; NB provides predict_proba
    y_score = None
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_test_text)
        if probs.ndim == 2 and probs.shape[1] == 2:
            # Positive class is label 1 (by encoding); this assumes binary labels
            y_score = probs[:, 1]
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X_test_text)
        if np.ndim(scores) == 1:
            y_score = scores

    acc = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred,
        average="binary",
        pos_label=1,
        zero_division=0
    )
    cm = confusion_matrix(y_test, y_pred)

    roc_auc = np.nan
    if y_score is not None and len(np.unique(y_test)) == 2:
        try:
            roc_auc = roc_auc_score(y_test, y_score)
        except Exception:
            pass

    print(f"\n{'=' * 70}")
    print(name)
    print(f"{'=' * 70}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")
    if not np.isnan(roc_auc):
        print(f"ROC-AUC  : {roc_auc:.4f}")

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=4, zero_division=0))

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "cm": cm,
        "y_true": y_test,
        "y_pred": y_pred,
        "trained_model": model
    }

In [6]:
# =========================================
# 8) Run experiments
# =========================================

models = build_models()
results = {}

for model_name, model in models.items():
    results[model_name] = evaluate_model(
        model=clone(model),
        X_train_text=X_train_text,
        y_train=y_train,
        X_test_text=X_test_text,
        y_test=y_test,
        name=f"New paired dataset - {model_name}"
    )

# =========================================
# 9) Summarize results
# =========================================

results_df = pd.DataFrame([
    {
        "model": m,
        "accuracy": r["accuracy"],
        "precision": r["precision"],
        "recall": r["recall"],
        "f1": r["f1"],
        "roc_auc": r["roc_auc"]
    }
    for m, r in results.items()
]).sort_values("f1", ascending=False).reset_index(drop=True)

print("\n===== All Results on new dataset (TF-IDF) =====")
print(results_df.round(4))

print("\n===== Best model (by F1) =====")
print(results_df.iloc[0].to_dict())


New paired dataset - TFIDF+MultinomialNB
Accuracy : 0.9918
Precision: 1.0000
Recall   : 0.9836
F1-score : 0.9917
ROC-AUC  : 0.9956

Confusion Matrix:
[[304   0]
 [  5 299]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9838    1.0000    0.9918       304
           1     1.0000    0.9836    0.9917       304

    accuracy                         0.9918       608
   macro avg     0.9919    0.9918    0.9918       608
weighted avg     0.9919    0.9918    0.9918       608


New paired dataset - TFIDF+LinearSVC
Accuracy : 0.9918
Precision: 1.0000
Recall   : 0.9836
F1-score : 0.9917
ROC-AUC  : 0.9950

Confusion Matrix:
[[304   0]
 [  5 299]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9838    1.0000    0.9918       304
           1     1.0000    0.9836    0.9917       304

    accuracy                         0.9918       608
   macro avg     0.9919    0.9918    0.9918       608
weighted 

# analysis what happend

In [7]:
# =========================================
# Robust evaluation + ablation + artifact probe
# For paired Persian human vs machine dataset
# =========================================

import os
import re
import warnings
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import LabelEncoder, FunctionTransformer, MaxAbsScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

warnings.filterwarnings("ignore")

# =========================================
# 1) Load data
# =========================================

DATA_PATH = "/kaggle/input/datasets/aabdollahii/humanvsai/dataset (1).xlsx"
assert os.path.exists(DATA_PATH), f"File not found: {DATA_PATH}"

df = pd.read_excel(DATA_PATH)
print("Original shape:", df.shape)
print("Columns:", df.columns.tolist())

# =========================================
# 2) Cleaning
# =========================================

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"[\u200b-\u200f\uFEFF]", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df = df.reset_index().rename(columns={"index": "group_id"})
df["text_clean"] = df["text"].apply(clean_text)
df["machine_text_clean"] = df["machine_text"].apply(clean_text)

df = df[
    (df["text_clean"].str.len() > 0) &
    (df["machine_text_clean"].str.len() > 0)
].reset_index(drop=True)

print("After cleaning:", df.shape)

# =========================================
# 3) Convert paired rows to long format
# =========================================

human_part = df[["group_id", "year", "filename", "text_clean", "label"]].copy()
human_part = human_part.rename(columns={"text_clean": "text", "label": "target"})
human_part["source_type"] = "human"

machine_part = df[["group_id", "year", "filename", "machine_text_clean", "machine_label"]].copy()
machine_part = machine_part.rename(columns={"machine_text_clean": "text", "machine_label": "target"})
machine_part["source_type"] = "machine"

df_long = pd.concat([human_part, machine_part], ignore_index=True)
df_long = df_long[df_long["text"].str.len() > 0].reset_index(drop=True)

print("Long format shape:", df_long.shape)
print(df_long["target"].value_counts(dropna=False))

# =========================================
# 4) Encode labels
# =========================================

le = LabelEncoder()
y = le.fit_transform(df_long["target"].astype(str).values)
X_text = df_long["text"].astype(str).values
groups = df_long["group_id"].values

print("Label mapping:")
for cls, idx in zip(le.classes_, range(len(le.classes_))):
    print(f"  {cls} -> {idx}")

# Try to identify positive class as machine-like
positive_class = 1

# =========================================
# 5) Surface feature extractor for artifact probe
# =========================================

persian_digits = "۰۱۲۳۴۵۶۷۸۹"
latin_digits = "0123456789"
all_digits = set(persian_digits + latin_digits)
puncts = set(".,!?؟؛:،\"'()[]{}-/\\@#$%^&*_+=<>|~`")

def extract_surface_features(texts):
    rows = []
    for text in texts:
        text = str(text)
        chars = list(text)
        tokens = text.split()

        char_count = len(text)
        token_count = len(tokens)
        unique_token_count = len(set(tokens))
        avg_token_len = np.mean([len(t) for t in tokens]) if tokens else 0.0
        type_token_ratio = (unique_token_count / token_count) if token_count > 0 else 0.0

        digit_count = sum(ch in all_digits for ch in chars)
        punct_count = sum(ch in puncts for ch in chars)
        space_count = sum(ch.isspace() for ch in chars)

        comma_count = text.count("،") + text.count(",")
        question_count = text.count("؟") + text.count("?")
        colon_count = text.count(":")
        semicolon_count = text.count("؛") + text.count(";")
        quote_count = text.count('"') + text.count("'")
        newline_count = text.count("\n")

        arabic_yeh = text.count("ي")
        persian_yeh = text.count("ی")
        arabic_kaf = text.count("ك")
        persian_kaf = text.count("ک")
        zwnj_count = text.count("\u200c")

        rows.append([
            char_count,
            token_count,
            unique_token_count,
            avg_token_len,
            type_token_ratio,
            digit_count,
            punct_count,
            space_count,
            comma_count,
            question_count,
            colon_count,
            semicolon_count,
            quote_count,
            newline_count,
            arabic_yeh,
            persian_yeh,
            arabic_kaf,
            persian_kaf,
            zwnj_count,
        ])
    return np.array(rows, dtype=float)

surface_transformer = FunctionTransformer(extract_surface_features, validate=False)

# =========================================
# 6) Define experiments
# =========================================

experiments = {
    "word_unigram_nb": Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 1),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            lowercase=False
        )),
        ("clf", MultinomialNB(alpha=1.0))
    ]),

    "word_uni_bigram_svc": Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            lowercase=False
        )),
        ("clf", CalibratedClassifierCV(
            estimator=LinearSVC(C=1.0),
            method="sigmoid",
            cv=3
        ))
    ]),

    "char_3_5_svc": Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="char",
            ngram_range=(3, 5),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            lowercase=False
        )),
        ("clf", CalibratedClassifierCV(
            estimator=LinearSVC(C=1.0),
            method="sigmoid",
            cv=3
        ))
    ]),

    "charwb_3_5_svc": Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            lowercase=False
        )),
        ("clf", CalibratedClassifierCV(
            estimator=LinearSVC(C=1.0),
            method="sigmoid",
            cv=3
        ))
    ]),

    "surface_only_logreg": Pipeline([
        ("surface", surface_transformer),
        ("scale", MaxAbsScaler()),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight=None,
            random_state=42
        ))
    ]),

    "word_bigram_plus_surface": Pipeline([
        ("features", FeatureUnion([
            ("word_tfidf", TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True,
                lowercase=False
            )),
            ("surface", Pipeline([
                ("extract", surface_transformer),
                ("scale", MaxAbsScaler())
            ]))
        ])),
        ("clf", CalibratedClassifierCV(
            estimator=LinearSVC(C=1.0),
            method="sigmoid",
            cv=3
        ))
    ]),
}

# =========================================
# 7) Cross-validation evaluation
# =========================================

def evaluate_cv(model, X, y, groups, n_splits=5, random_state=42):
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    fold_rows = []
    all_true = []
    all_pred = []
    all_prob = []

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        clf = clone(model)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        if hasattr(clf, "predict_proba"):
            y_score = clf.predict_proba(X_test)[:, 1]
        else:
            y_score = None

        acc = accuracy_score(y_test, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_test, y_pred,
            average="binary",
            pos_label=positive_class,
            zero_division=0
        )

        roc_auc = np.nan
        if y_score is not None and len(np.unique(y_test)) == 2:
            try:
                roc_auc = roc_auc_score(y_test, y_score)
            except Exception:
                pass

        fold_rows.append({
            "fold": fold,
            "accuracy": acc,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "roc_auc": roc_auc
        })

        all_true.extend(y_test.tolist())
        all_pred.extend(y_pred.tolist())
        if y_score is not None:
            all_prob.extend(y_score.tolist())

    fold_df = pd.DataFrame(fold_rows)

    summary = {
        "accuracy_mean": fold_df["accuracy"].mean(),
        "accuracy_std": fold_df["accuracy"].std(ddof=1),
        "precision_mean": fold_df["precision"].mean(),
        "precision_std": fold_df["precision"].std(ddof=1),
        "recall_mean": fold_df["recall"].mean(),
        "recall_std": fold_df["recall"].std(ddof=1),
        "f1_mean": fold_df["f1"].mean(),
        "f1_std": fold_df["f1"].std(ddof=1),
        "roc_auc_mean": fold_df["roc_auc"].mean(skipna=True),
        "roc_auc_std": fold_df["roc_auc"].std(ddof=1, skipna=True),
    }

    return fold_df, summary, np.array(all_true), np.array(all_pred)

# =========================================
# 8) Run all experiments
# =========================================

all_summaries = []
all_fold_tables = {}

for name, model in experiments.items():
    print("\n" + "=" * 80)
    print("Running:", name)
    print("=" * 80)

    fold_df, summary, y_true_all, y_pred_all = evaluate_cv(model, X_text, y, groups, n_splits=5)

    print(fold_df.round(4))
    print("\nSummary:")
    for k, v in summary.items():
        print(f"{k}: {v:.4f}" if pd.notna(v) else f"{k}: NaN")

    all_fold_tables[name] = fold_df
    row = {"experiment": name}
    row.update(summary)
    all_summaries.append(row)

summary_df = pd.DataFrame(all_summaries).sort_values("f1_mean", ascending=False).reset_index(drop=True)

print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)
print(summary_df.round(4))

# =========================================
# 9) Optional: inspect top terms for a simple linear model
# =========================================

print("\n" + "=" * 80)
print("Top feature inspection for word_uni_bigram_svc alternative using LogisticRegression")
print("=" * 80)

inspect_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        lowercase=False
    )),
    ("clf", LogisticRegression(max_iter=3000, random_state=42))
])

inspect_model.fit(X_text, y)
vectorizer = inspect_model.named_steps["tfidf"]
clf = inspect_model.named_steps["clf"]

feature_names = np.array(vectorizer.get_feature_names_out())
coefs = clf.coef_[0]

top_machine_idx = np.argsort(coefs)[-30:][::-1]
top_human_idx = np.argsort(coefs)[:30]

print("\nTop features pushing toward class 1:")
for idx in top_machine_idx:
    print(f"{feature_names[idx]:<30} {coefs[idx]:.4f}")

print("\nTop features pushing toward class 0:")
for idx in top_human_idx:
    print(f"{feature_names[idx]:<30} {coefs[idx]:.4f}")


Original shape: (1519, 10)
Columns: ['year', 'filename', 'text', 'word_count', 'label', 'machine_text', 'machine_label', 'generation_status', 'generation_error', 'processed_at']
After cleaning: (1519, 13)
Long format shape: (3038, 6)
target
human      1519
machine    1519
Name: count, dtype: int64
Label mapping:
  human -> 0
  machine -> 1

Running: word_unigram_nb
   fold  accuracy  precision  recall      f1  roc_auc
0     1    0.9852        1.0  0.9704  0.9850   0.9934
1     2    0.9901        1.0  0.9803  0.9900   0.9967
2     3    0.9918        1.0  0.9836  0.9917   0.9923
3     4    0.9868        1.0  0.9736  0.9866   0.9944
4     5    0.9885        1.0  0.9770  0.9884   0.9987

Summary:
accuracy_mean: 0.9885
accuracy_std: 0.0026
precision_mean: 1.0000
precision_std: 0.0000
recall_mean: 0.9770
recall_std: 0.0052
f1_mean: 0.9883
f1_std: 0.0027
roc_auc_mean: 0.9951
roc_auc_std: 0.0026

Running: word_uni_bigram_svc
   fold  accuracy  precision  recall      f1  roc_auc
0     1    0.98